In [1]:
import pandas as pd
import yfinance as yf
import datetime, time
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import requests
import zipfile
import io
import statsmodels.api as sm
import os
import glob
from sqlalchemy import create_engine, types as satypes
import contextlib
import xml.etree.ElementTree as ET
from tqdm import tqdm


In [23]:
folder_path = os.path.join(os.path.expanduser("~"), "Desktop")
os.path.samefile(folder_path, os.getcwd()), time.strftime("%Y-%m-%d %H:%M:%S", time.localtime())

(True, '2025-10-05 01:28:24')

In [2]:
username = "postgres"
password = "GeorgeHighbury0725!!"
host = "localhost"
port = "5432"
database = "sp_500_OHLCV_database"

In [3]:
engine = create_engine(f"postgresql://{username}:{password}@{host}:{port}/{database}")
connection = engine.connect()

In [ ]:
sp500_url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"

try:
    wikipedia_tables = pd.read_html(sp500_url)
except Exception:
    headers = {"User-Agent": "Mozilla/5.0"}
    response = requests.get(sp500_url, headers=headers)
    response.raise_for_status()
    wikipedia_tables = pd.read_html(io.StringIO(response.text))

wikipedia_table = wikipedia_tables[0]

wikipedia_table = wikipedia_tables[0].rename(columns={
    "Symbol": "ticker",
    "Security": "company_name",
    "GICS Sector": "sector",
    "GICS Sub-Industry": "sub_industry",
    "Headquarters Location": "headquarters_location",
    "Date added": "date_added",
    "CIK": "cik",
    "Founded": "founded"
})

wikipedia_table.to_sql("sp500_reference", engine, if_exists="replace", index=False)

In [4]:
sp500_reference = pd.read_sql("SELECT * FROM sp500_reference;", engine)
sp500_reference

,ticker,company_name,sector,sub_industry,headquarters_location,date_added,cik,founded
0,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",1957-03-04,66740,1902
1,AOS,A. O. Smith,Industrials,Building Products,"Milwaukee, Wisconsin",2017-07-26,91142,1916
2,ABT,Abbott Laboratories,Health Care,Health Care Equipment,"North Chicago, Illinois",1957-03-04,1800,1888
3,ABBV,AbbVie,Health Care,Biotechnology,"North Chicago, Illinois",2012-12-31,1551152,2013 (1888)
4,ACN,Accenture,Information Technology,IT Consulting & Other Services,"Dublin, Ireland",2011-07-06,1467373,1989
...,...,...,...,...,...,...,...,...
498,XYL,Xylem Inc.,Industrials,Industrial Machinery & Supplies & Components,"White Plains, New York",2011-11-01,1524472,2011
499,YUM,Yum! Brands,Consumer Discretionary,Restaurants,"Louisville, Kentucky",1997-10-06,1041061,1997
500,ZBRA,Zebra Technologies,Information Technology,Electronic Equipment & Instruments,"Lincolnshire, Illinois",2019-12-23,877212,1969
501,ZBH,Zimmer Biomet,Health Care,Health Care Equipment,"Warsaw, Indiana",2001-08-07,1136869,1927


In [5]:
symbol_column = sp500_reference["ticker"]

ticker_list = []
for ticker in symbol_column:
    ticker_list.append(ticker)

ticker_list = [ticker.replace(".", "-") for ticker in ticker_list]  # Wikipedia uses "." in some tickers to denote class A/B shares. Also this is an example of list comprehension.
ticker_list.sort()

In [9]:
start_input = input("Start date (YYYY-MM-DD or 'start'): ").lower()
if start_input == "start":
    start = pd.Timestamp.now().normalize()
else:
    start = pd.Timestamp(datetime.datetime.strptime(start_input, "%Y-%m-%d")).normalize()
end_input = input("End date (YYYY-MM-DD or 'now'): ").lower()
if end_input == "now":
    end = pd.Timestamp.now().normalize()
else:
    end = pd.Timestamp(datetime.datetime.strptime(end_input, "%Y-%m-%d")).normalize()

In [24]:
with engine.connect() as conn:
    ohlcv_dates = pd.read_sql("""
        SELECT 
            MIN(date) AS earliest_date,
            MAX(date) AS latest_date
        FROM sp500_ohlcv;
    """, conn).iloc[0]

    edgar_dates = pd.read_sql("""
        SELECT 
            MIN(filed) AS earliest_date,
            MAX(filed) AS latest_date
        FROM sp500_edgar_financials;
    """, conn).iloc[0]

ohlcv_start = pd.to_datetime(ohlcv_dates["earliest_date"])
ohlcv_end   = pd.to_datetime(ohlcv_dates["latest_date"])
edgar_start = pd.to_datetime(edgar_dates["earliest_date"])
edgar_end   = pd.to_datetime(edgar_dates["latest_date"])

ohlcv_start, ohlcv_end, edgar_start, edgar_end

(Timestamp('2025-01-02 00:00:00'),
 Timestamp('2025-10-03 00:00:00'),
 Timestamp('2009-04-15 00:00:00'),
 Timestamp('2025-10-01 00:00:00'))

In [ ]:
sp500df_ohlcv_append_raw = yf.download(ticker_list,start=sp500df.index.max()+ pd.Timedelta(days=1), end=end, group_by="ticker", auto_adjust=False, threads=False)

In [11]:
sp500df_ohlcv_append_flat = (
    sp500df_ohlcv_append_raw
    .stack(level=0, future_stack=True)
    .rename_axis(["date", "ticker"])
    .reset_index()
    .rename(columns={
        "Date": "date",
        "Ticker": "ticker",
        "Open": "open",
        "High": "high",
        "Low": "low",
        "Close": "close",
        "Adj Close": "adj_close",
        "Volume": "volume"
    })
    .query("date < @end")
    .sort_values(["ticker", "date"])
    .reset_index(drop=True)
)

sp500df_ohlcv_append_flat.columns.name = None

sp500df_ohlcv_append_flat.tail()

,date,ticker,open,high,low,close,adj_close,volume
498,2025-10-03,XYZ,76.690002,78.245003,76.099998,76.949997,76.949997,5755700
499,2025-10-03,YUM,150.990005,151.229996,149.830002,150.690002,150.690002,1288500
500,2025-10-03,ZBH,99.180000,101.110001,99.129997,100.779999,100.779999,896200
501,2025-10-03,ZBRA,299.109985,311.440002,298.399994,305.010010,305.010010,452300
502,2025-10-03,ZTS,146.720001,148.789993,146.250000,146.419998,146.419998,2569700


In [12]:
sp500df_ohlcv_append_flat.to_sql("sp500_ohlcv", engine, if_exists="append", index=False)

503

In [13]:
sp500df = pd.read_sql("SELECT * FROM sp500_ohlcv;", engine, parse_dates=["date"]).pivot(index="date", columns="ticker", values=["open", "high", "low", "close", "adj_close", "volume"]).sort_index(axis=1, level=0)
sp500df.tail()

adj_close                                                  \
ticker               A        AAPL        ABBV        ABNB         ABT   
date                                                                     
2025-09-29  123.750000  254.429993  223.160004  122.919998  133.110001   
2025-09-30  128.350006  254.630005  231.539993  121.419998  133.940002   
2025-10-01  138.580002  255.449997  244.380005  122.320000  133.470001   
2025-10-02  138.699997  257.130005  236.559998  121.489998  132.990005   
2025-10-03  141.639999  258.019989  233.910004  120.220001  134.589996   

                                                                      ...  \
ticker           ACGL         ACN        ADBE         ADI        ADM  ...   
date                                                                  ...   
2025-09-29  89.830002  247.000000  359.420013  244.789993  60.310001  ...   
2025-09-30  90.730003  246.600006  352.750000  245.699997  59.740002  ...   
2025-10-01  90.309998  243.710007  343.720001  239.279999  59.250000  ...   
2025-10-02  89.080002  244.339996  351.480011  241.669998  59.110001  ...   
2025-10-03  90.790001  245.320007  346.739990  241.990005  61.040001  ...   

               volume                                                          \
ticker             WY       WYNN        XEL         XOM        XYL        XYZ   
date                                                                            
2025-09-29  5111100.0  2014800.0  5527100.0  19189300.0  1087600.0  7237200.0   
2025-09-30  5465100.0  1511300.0  4011000.0  18076200.0  1841000.0  7838100.0   
2025-10-01  3650300.0  1532600.0  4585600.0  16614000.0  1321000.0  6640700.0   
2025-10-02  3592700.0  1286100.0  6982500.0  13059400.0  1347100.0  7852600.0   
2025-10-03  2943500.0  3619500.0  3850200.0  12948600.0  1245600.0  5755700.0   

                                                       
ticker            YUM        ZBH      ZBRA        ZTS  
date                                                   
2025-09-29  1742400.0   929000.0  482300.0  2870100.0  
2025-09-30  1898800.0  1188300.0  500700.0  3736800.0  
2025-10-01  1991500.0  1193600.0  537300.0  3677000.0  
2025-10-02  1800700.0   730500.0  450100.0  3262300.0  
2025-10-03  1288500.0   896200.0  452300.0  2569700.0  

[5 rows x 3048 columns]

In [14]:
daily_returns = sp500df["adj_close"].pct_change(fill_method=None).dropna(how="all")
daily_returns.tail()

ticker,A,AAPL,ABBV,ABNB,ABT,ACGL,ACN,ADBE,ADI,ADM,...,WY,WYNN,XEL,XOM,XYL,XYZ,YUM,ZBH,ZBRA,ZTS
date,,,,,,,,,,,,,,,,,,,,,
2025-09-29,0.002918,-0.004032,0.011559,-0.006306,-0.003295,-0.011554,0.033603,-0.002636,-0.011189,-0.004785,...,0.013040,0.026828,0.009712,-0.025593,0.006432,0.023347,0.008121,0.000407,-0.009507,-0.003066
2025-09-30,0.037172,0.000786,0.037551,-0.012203,0.006235,0.010019,-0.001619,-0.018558,0.003717,-0.009451,...,-0.002816,-0.031413,0.007495,-0.012870,0.024661,-0.041385,-0.012602,0.002239,0.000808,0.022788
2025-10-01,0.079704,0.003220,0.055455,0.007412,-0.003509,-0.004629,-0.011719,-0.025599,-0.026129,-0.008202,...,0.006454,0.029313,-0.004216,-0.006741,0.002576,0.015636,0.007829,0.003959,-0.018071,0.004306
2025-10-02,0.000866,0.006577,-0.031999,-0.006785,-0.003596,-0.013620,0.002585,0.022577,0.009988,-0.002363,...,-0.000802,0.009922,-0.008841,-0.006251,0.007641,0.046458,-0.012011,0.002831,0.013914,-0.003130
2025-10-03,0.021197,0.003461,-0.011202,-0.010454,0.012031,0.019196,0.004011,-0.013486,0.001324,0.032651,...,0.007621,-0.072596,0.008292,0.017702,0.005637,0.001823,-0.004361,0.016235,0.030962,-0.000478


In [15]:
returns_1d = daily_returns.iloc[-1].dropna().sort_values(ascending=False)
returns_1d.head(10), returns_1d.tail(10)

(ticker
 HUM     0.105604
 CNC     0.051059
 CI      0.047209
 EBAY    0.042647
 MOH     0.039743
 BEN     0.038848
 CHTR    0.038574
 CMG     0.037000
 FICO    0.036965
 F       0.036825
 Name: 2025-10-03 00:00:00, dtype: float64,
 ticker
 MCK    -0.027204
 PM     -0.029384
 AMD    -0.029812
 KLAC   -0.033100
 NKE    -0.035403
 DELL   -0.044989
 JBL    -0.063100
 WYNN   -0.072596
 LVS    -0.074114
 PLTR   -0.074739
 Name: 2025-10-03 00:00:00, dtype: float64)

In [ ]:
lookback_map = {
    "1D": 1,
    "1W": 5,
    "1M": 21,
    "3M": 63,
    "6M": 126
}

choice = input("Enter lookback horizon (1D, 1W, 1M, 3M, 6M, ALL, or number of trading days): ").upper()

if choice in lookback_map:
    window = lookback_map[choice]
    returns = (
        sp500df["adj_close"]
        .pct_change(window, fill_method=None)
        .iloc[-1]
        .dropna()
        .sort_values(ascending=False)
    )
    display(returns.head(10), returns.tail(10))

elif choice == "ALL":
    full_cum_returns = (
        sp500df["adj_close"].iloc[-1] / sp500df["adj_close"].iloc[0] - 1
    ).dropna().sort_values(ascending=False)
    display(full_cum_returns.head(10), full_cum_returns.tail(10))

elif choice.isdigit():
    window = int(choice)
    if len(sp500df) > window:
        returns = (
            sp500df["adj_close"]
            .pct_change(window, fill_method=None)
            .iloc[-1]
            .dropna()
            .sort_values(ascending=False)
        )
        display(returns.head(10), returns.tail(10))
    else:
        print(f"Not enough rows in dataset for {window} trading days.")

else:
    print("Invalid choice. Please enter 1D, 1W, 1M, 3M, 6M, ALL, or a number of trading days.")

In [ ]:
fig = go.Figure()

for ticker in daily_returns.columns:
    fig.add_trace(go.Scatter(
        x=daily_returns.index,
        y=daily_returns[ticker],
        mode='lines',
        name=ticker,
        customdata=[[ticker]] * len(daily_returns),
        hovertemplate=(
        "Date: %{x}<br>" +
        "Return: %{y:.4f}<br>" +
        "Ticker: %{customdata[0]}<extra></extra>"
        )
    ))

fig.update_layout(
    title='Daily Returns of S&P 500 Tickers',
    xaxis_title='Date',
    yaxis_title='Daily Return',
    template='plotly_dark',
    xaxis=dict(rangeslider=dict(visible=True))
)

fig.show(config={'displaylogo': False})

In [16]:
user_input_stock_symbol = input("Enter stock symbol to inspect (e.g., 'AAPL', 'MSFT'): ").upper()

open = sp500df["open"][user_input_stock_symbol]
high = sp500df["high"][user_input_stock_symbol]
low = sp500df["low"][user_input_stock_symbol]
close = sp500df["close"][user_input_stock_symbol]
volume = sp500df["volume"][user_input_stock_symbol]
adj_close = sp500df["adj_close"][user_input_stock_symbol]

ticker = pd.DataFrame({
    "date": adj_close.index,
    "open": open.values,
    "high": high.values,
    "low": low.values,
    "close": close.values,
    "adj_close": adj_close.values,
    "volume": volume.values,
})

ticker["returns"] = ticker["adj_close"].pct_change(fill_method=None)
ticker.reset_index(drop=True, inplace=True)
ticker.tail()

,date,open,high,low,close,adj_close,volume,returns
184,2025-09-29,169.309998,170.100006,164.660004,165.339996,165.339996,2686400.0,-0.019161
185,2025-09-30,165.399994,165.830002,161.050003,161.949997,161.949997,2449000.0,-0.020503
186,2025-10-01,161.679993,165.449997,160.570007,161.910004,161.910004,3232900.0,-0.000247
187,2025-10-02,162.970001,168.580002,162.479996,167.300003,167.300003,2641900.0,0.033290
188,2025-10-03,168.229996,172.210007,165.830002,166.279999,166.279999,2270000.0,-0.006097


In [17]:
lookback_map = {
    "1D": 1,
    "1W": 5,
    "1M": 21,
    "3M": 63,
    "6M": 126
}

choice = input("Enter lookback horizon (1D, 1W, 1M, 3M, 6M, ALL, or number of trading days): ").upper()

if choice in lookback_map:
    window = lookback_map[choice]
    if len(ticker) > window:
        ret = ticker["adj_close"].iloc[-1] / ticker["adj_close"].iloc[-window-1] - 1
        print(f"{choice} return for {user_input_stock_symbol}: {ret:.2%}")
    else:
        print(f"Not enough data for {choice}")

elif choice == "ALL":
    ret = ticker["adj_close"].iloc[-1] / ticker["adj_close"].iloc[0] - 1
    print(f"Total return for {user_input_stock_symbol}: {ret:.2%}")

elif choice.isdigit():
    window = int(choice)
    if len(ticker) > window:
        ret = ticker["adj_close"].iloc[-1] / ticker["adj_close"].iloc[-window-1] - 1
        print(f"{window}-day return for {user_input_stock_symbol}: {ret:.2%}")
    else:
        print(f"Not enough data for {window} trading days")

else:
    print("Invalid choice. Please enter 1D, 1W, 1M, 3M, 6M, ALL, or a number of trading days.")

100-day return for NRG: 10.70%


In [18]:
fig = go.Figure()

fig.add_trace(go.Candlestick(
    x=ticker["date"], open=ticker["open"], high=ticker["high"], low=ticker["low"], close=ticker["close"], name="Candlestick"
))

fig.add_trace(go.Bar(
    x=ticker["date"], y=ticker["volume"], name="Volume", marker=dict(color="gray"), opacity=0.3, yaxis="y2"
))

fig.update_layout(
    title=f"{user_input_stock_symbol} Candlestick and Volume",
    xaxis=dict(title="Date", tickformat="%b %d"),
    yaxis=dict(title="Price ($)"),
    yaxis2=dict(title="Volume", overlaying="y", side="right", showgrid=False, color="gray"),
    legend=dict(x=0.01, y=0.99, bordercolor="black", borderwidth=1),
    bargap=0,
    template="plotly_dark",
    width=1200,
    height=500
)

fig.show(config={'displaylogo': False})


In [19]:
fig = make_subplots(rows=1, cols=2, subplot_titles=(
    f"{user_input_stock_symbol} Daily Returns 2025",
    f"{user_input_stock_symbol} Daily Returns Distribution"
))

fig.add_trace(
    go.Scatter(x=ticker["date"], y=ticker["returns"], mode="lines+markers",
               line=dict(color="blue", width=0.5),
               marker=dict(symbol="triangle-down", size=4),
               name="Daily Returns"),
    row=1, col=1
)

fig.add_trace(
    go.Histogram(x=ticker["returns"].dropna(), nbinsx=25,
                 marker=dict(color="blue", line=dict(color="black", width=1)),
                 opacity=0.7, name="Distribution"),
    row=1, col=2
)

fig.update_layout(
    template="plotly_dark",
    width=1200, height=500,
    showlegend=False
)

fig.update_xaxes(title_text="Date", tickformat="%b %d", row=1, col=1)
fig.update_yaxes(title_text="% Change", row=1, col=1)

fig.update_xaxes(title_text="Daily Return (% Change)", row=1, col=2)
fig.update_yaxes(title_text="Frequency", row=1, col=2)

fig.show(config={'displaylogo': False})

In [ ]:
fasb_fetch_year = datetime.datetime.now().year
url = f"https://xbrl.fasb.org/us-gaap/{fasb_fetch_year}/us-gaap-{fasb_fetch_year}.zip"

resp = requests.get(url)
resp.raise_for_status()

elements = []

with zipfile.ZipFile(io.BytesIO(resp.content)) as z:
    xsd_files = [name for name in z.namelist() if name.startswith(f"us-gaap-{fasb_fetch_year}/elts/") and name.endswith(".xsd")]
    
    for xsd_file in xsd_files:
        with z.open(xsd_file) as f:
            try:
                tree = ET.parse(f)
                root = tree.getroot()
                for elem in root.findall(".//{http://www.w3.org/2001/XMLSchema}element"):
                    elements.append({
                        "id": elem.get("id"),
                        "name": elem.get("name"),
                        "type": elem.get("type"),
                        "substitutionGroup": elem.get("substitutionGroup"),
                        "balance": elem.get("balance"),
                        "periodType": elem.get("periodType"),
                        "source": xsd_file
                    })
            except Exception:
                # Skip non-XML or problematic files
                continue

xbrl_taxonomy = pd.DataFrame(elements).drop_duplicates(subset=["id"])

In [ ]:
xbrl_taxonomy.to_sql("xbrl_taxonomy", engine, if_exists="replace", index=False)

In [25]:
xbrl_taxonomy = pd.read_sql("SELECT * FROM xbrl_taxonomy;", engine)
xbrl_taxonomy

,id,name,type,substitutionGroup,balance,periodType
0,us-gaap_AccidentAndHealthInsuranceSegmentMember,AccidentAndHealthInsuranceSegmentMember,dtr-types:domainItemType,xbrli:item,None,None
1,us-gaap_OtherAccountsPayableAndAccruedLiabilities,OtherAccountsPayableAndAccruedLiabilities,xbrli:monetaryItemType,xbrli:item,None,None
2,us-gaap_AccountingForCertainLoansAndDebtSecuri...,AccountingForCertainLoansAndDebtSecuritiesAcqu...,dtr-types:textBlockItemType,xbrli:item,None,None
3,us-gaap_InterestsContinuedToBeHeldByTransferor...,InterestsContinuedToBeHeldByTransferorInFinanc...,xbrli:stringItemType,xbrli:item,None,None
4,us-gaap_InterestsContinuedToBeHeldByTransferor...,InterestsContinuedToBeHeldByTransferorInFinanc...,dtr-types:textBlockItemType,xbrli:item,None,None
...,...,...,...,...,...,...
17330,us-gaap_ShareBasedPaymentArrangementValuationT...,ShareBasedPaymentArrangementValuationTechnique...,dtr-types:domainItemType,xbrli:item,None,None
17331,us-gaap_ShareBasedPaymentArrangementMeasuremen...,ShareBasedPaymentArrangementMeasurementInputDo...,dtr-types:domainItemType,xbrli:item,None,None
17332,us-gaap_TrinomialModelMember,TrinomialModelMember,dtr-types:domainItemType,xbrli:item,None,None
17333,us-gaap_InvestmentCompanySupplementalIncomeAbs...,InvestmentCompanySupplementalIncomeAbstract,xbrli:stringItemType,xbrli:item,None,None


In [17]:
xbrl_query_ids = [
    "us-gaap_Assets",
    "us-gaap_AssetsCurrent",
    "us-gaap_AssetsNoncurrent",

    "us-gaap_Liabilities",
    "us-gaap_LiabilitiesCurrent",
    "us-gaap_LiabilitiesNoncurrent",

    "us-gaap_StockholdersEquity",
    "us-gaap_StockholdersEquityIncludingPortionAttributableToNoncontrollingInterest",
    "us-gaap_LiabilitiesAndStockholdersEquity",

    "us-gaap_RevenueFromContractWithCustomerExcludingAssessedTax",
    "us-gaap_SalesRevenueNet",
    "us-gaap_Revenues",
    "us-gaap_RevenueFromContractWithCustomerIncludingAssessedTax",

    "us-gaap_GrossProfit",
    "us-gaap_GrossProfitAbstract",

    "us-gaap_OperatingIncomeLoss",

    "us-gaap_NetIncomeLoss",
    "us-gaap_ProfitLoss",
    "us-gaap_NetIncomeLossAvailableToCommonStockholdersBasic",
    "us-gaap_NetIncomeLossAvailableToCommonStockholdersDiluted",
    "us-gaap_NetIncomeLossAttributableToNoncontrollingInterest",
    "us-gaap_NetIncomeLossAttributableToParent",
    "us-gaap_NetIncomeLossAttributableToRedeemableNoncontrollingInterest",
    "us-gaap_NetIncomeLossAttributableToNonredeemableNoncontrollingInterest",
    "us-gaap_NetIncomeLossIncludingPortionAttributableToNoncontrollingInterest",
    "us-gaap_NetIncomeLossAbstract",

    "us-gaap_EarningsPerShareBasic",
    "us-gaap_EarningsPerShareDiluted",

    "us-gaap_NetCashProvidedByUsedInOperatingActivities",
    "us-gaap_NetCashProvidedByUsedInInvestingActivities",
    "us-gaap_NetCashProvidedByUsedInFinancingActivities",
    "us-gaap_PaymentsToAcquirePropertyPlantAndEquipment",

    "us-gaap_WeightedAverageNumberOfSharesOutstandingBasic",
    "us-gaap_WeightedAverageNumberOfDilutedSharesOutstanding"
]

In [ ]:
with engine.connect() as conn:
    last_filed = pd.read_sql(
        "SELECT MAX(filed) as max_date FROM sp500_edgar_financials", conn
    ).iloc[0]["max_date"]

last_filed = pd.to_datetime(last_filed, errors="coerce").normalize()

id_to_name = dict(zip(xbrl_taxonomy["id"], xbrl_taxonomy["name"]))

sp500_edgar_financials_append = []

pbar = tqdm(sp500_reference.iterrows(), total=len(sp500_reference), desc="Updating financial metrics")

for _, row in pbar:
    ticker = row["ticker"]
    cik = str(row["cik"]).zfill(10)
    
    url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{cik}.json"
    r = requests.get(url, headers={"User-Agent": "dbater1993@gmail.com"})
    if r.status_code != 200:
        continue
    
    financial_metrics = r.json().get("facts", {}).get("us-gaap", {})
    
    for xid in xbrl_query_ids:
        xname = id_to_name.get(xid, xid.replace("us-gaap_", ""))
        
        metric_data = financial_metrics.get(xname)
        if not metric_data:
            continue
        
        for unit, datapoints in metric_data.get("units", {}).items():
            for dp in datapoints:
                filed_date = pd.to_datetime(dp.get("filed"), errors="coerce").normalize()
                end_date = pd.to_datetime(dp.get("end"), errors="coerce").normalize()
                
                if pd.isna(filed_date):
                    continue  # skip if no valid filed date
                
                if pd.isna(last_filed) or filed_date > last_filed:
                    sp500_edgar_financials_append.append({
                        "ticker": ticker,
                        "cik": cik,
                        "id": xid,
                        "metric": xname,
                        "unit": unit,
                        "value": dp.get("val"),
                        "fy": dp.get("fy"),
                        "fp": dp.get("fp"),
                        "form": dp.get("form"),
                        "filed": filed_date,
                        "end": end_date
                    })
    
    pbar.set_description(f"Collected: {len(sp500_edgar_financials_append)} new rows")
    time.sleep(0.2)

pbar.close()

sp500_edgar_financials_append

In [32]:
sp500_edgar_financials_append = pd.DataFrame(sp500_edgar_financials_append)
sp500_edgar_financials_append.to_sql("sp500_edgar_financials", engine, if_exists="append", index=False)

75

In [20]:
user_stock_fundamentals_query = f"""
SELECT "end", id, metric, value
FROM sp500_edgar_financials
WHERE ticker = '{user_input_stock_symbol}' AND form = '10-K'
ORDER BY "end";
"""
user_stock_fundamentals_query_df = pd.read_sql(user_stock_fundamentals_query, engine)
user_stock_fundamentals_query_df

,end,id,metric,value
0,2006-12-31,us-gaap_StockholdersEquityIncludingPortionAttr...,StockholdersEquityIncludingPortionAttributable...,5.686000e+09
1,2007-12-31,us-gaap_StockholdersEquityIncludingPortionAttr...,StockholdersEquityIncludingPortionAttributable...,5.519000e+09
2,2007-12-31,us-gaap_StockholdersEquityIncludingPortionAttr...,StockholdersEquityIncludingPortionAttributable...,5.519000e+09
3,2007-12-31,us-gaap_PaymentsToAcquirePropertyPlantAndEquip...,PaymentsToAcquirePropertyPlantAndEquipment,4.810000e+08
4,2007-12-31,us-gaap_NetCashProvidedByUsedInFinancingActivi...,NetCashProvidedByUsedInFinancingActivities,-8.140000e+08
...,...,...,...,...
1735,2024-12-31,us-gaap_NetIncomeLoss,NetIncomeLoss,1.125000e+09
1736,2024-12-31,us-gaap_EarningsPerShareBasic,EarningsPerShareBasic,5.140000e+00
1737,2024-12-31,us-gaap_Revenues,Revenues,2.813000e+10
1738,2024-12-31,us-gaap_RevenueFromContractWithCustomerExcludi...,RevenueFromContractWithCustomerExcludingAssess...,2.774800e+10


In [ ]:
user_stock_fundamentals_query_df["end"] = pd.to_datetime(user_stock_fundamentals_query_df["end"])

pivot = user_stock_fundamentals_query_df.pivot_table(
    index="metric",
    columns=user_stock_fundamentals_query_df["end"].dt.year,
    values="value",
    aggfunc="last"
)

growth = (pivot - pivot.shift(axis=1)) / pivot.shift(axis=1).abs()
zmin, zmax = growth.quantile(0.05).min(), growth.quantile(0.95).max()

metrics = pivot.index.tolist()
metrics_sorted = sorted([m for m in metrics if m != "Assets"])
if "Assets" in metrics:
    metrics_sorted = ["Assets"] + metrics_sorted

pivot = pivot.loc[metrics_sorted]
growth = growth.loc[metrics_sorted]

fig = go.Figure(
    data=go.Heatmap(
        z=growth.fillna(0).values,
        x=growth.columns,
        y=growth.index,
        colorscale="Balance",
        zmin=-50,
        zmax=50,
        colorbar=dict(title="YoY % Growth"),
        customdata=pivot.fillna("").values,
        hovertemplate=(
            "<b>%{y}</b><br>"
            "Year: %{x}<br>"
            "Value: %{customdata:,}<br>"
            "Growth: %{z:.2f}%<extra></extra>"
        )
    )
)

fig.update_layout(
    title=f"{user_input_stock_symbol} Fundamentals Growth Heatmap (10-K)",
    xaxis_title="Year",
    yaxis_title="Metric",
    height=800,
    yaxis=dict(categoryorder="array", categoryarray=metrics_sorted)
)

fig.update_yaxes(autorange="reversed")

fig.show(config={'displaylogo': False})

In [ ]:
fama_french_url = 'https://mba.tuck.dartmouth.edu/pages/faculty/ken.french/ftp/F-F_Research_Data_5_Factors_2x3_daily_CSV.zip'

response = requests.get(fama_french_url)

with zipfile.ZipFile(io.BytesIO(response.content)) as z:
    file_name = z.namelist()[0]
    with z.open(file_name) as file:
        fama_french_five_factor = pd.read_csv(file,index_col=0, parse_dates=True,skiprows=3)

fama_french_five_factor = fama_french_five_factor.iloc[:-1]

fama_french_five_factor.reset_index(inplace=True)

fama_french_five_factor.columns = ['Date', 'Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA', 'RF']

fama_french_five_factor['Date'] = pd.to_datetime(fama_french_five_factor['Date'])

fama_french_five_factor = fama_french_five_factor[
    (fama_french_five_factor['Date'] >= start) &
    (fama_french_five_factor['Date'] <= end)
]

fama_french_five_factor.reset_index(drop=True, inplace=True)

fama_french_five_factor

In [ ]:
daily_returns.index = pd.to_datetime(daily_returns.index)

train_start = fama_french_five_factor['Date'].min()
train_end = fama_french_five_factor['Date'].max()

excess_returns = daily_returns.loc[
    daily_returns.index.intersection(fama_french_five_factor['Date'])
].sub(
    fama_french_five_factor.set_index("Date").loc[daily_returns.index.intersection(fama_french_five_factor['Date']), "RF"] / 100,
    axis=0
)

train_returns = excess_returns.loc[train_start:train_end]
train_factors = fama_french_five_factor.set_index("Date").loc[train_start:train_end, ['Mkt-RF', 'SMB', 'HML', 'RMW', 'CMA']]

regression_results = []

for ticker in train_returns.columns:
    y_train = train_returns[ticker]
    X_train = sm.add_constant(train_factors)

    common_idx = y_train.index.intersection(X_train.index)
    y_train = y_train.loc[common_idx]
    X_train = X_train.loc[common_idx]

    model = sm.OLS(y_train, X_train).fit()

    result = {
        'Ticker': ticker,
        'α': model.params['const'],
        'Mkt-RF': model.params.get('Mkt-RF', None),
        'SMB': model.params.get('SMB', None),
        'HML': model.params.get('HML', None),
        'RMW': model.params.get('RMW', None),
        'CMA': model.params.get('CMA', None),
        'R-squared': model.rsquared
    }
    regression_results.append(result)

regression_summary_df = pd.DataFrame(regression_results)

regression_summary_df.sort_values(by='α', ascending=False).reset_index(drop=True).head(10)


In [ ]:
def regression_sort():
    sort_by = input(f"Choose a column to sort by {list(regression_summary_df.columns[1:])}: ")
    order = input("Sort ascending? (yes/no): ").lower() == "yes"
    return regression_summary_df.sort_values(by=sort_by, ascending=order).reset_index(drop=True).head(10)

In [ ]:
y_actual = train_returns[user_input_stock_symbol]

X = sm.add_constant(train_factors)

common_idx = y_actual.index.intersection(X.index)

y_actual = y_actual.loc[common_idx]

X = X.loc[common_idx]

model = sm.OLS(y_actual, X).fit()

y_pred = model.predict(X)

fig, ax = plt.subplots(figsize=(30, 6))
ax.plot(y_actual.index, y_actual, label='Actual Excess Return', linewidth=1.5)
ax.plot(y_actual.index, y_pred, label='Predicted Excess Return (Line of Best Fit)', linestyle='--', linewidth=2)
ax.set_title(f'{user_input_stock_symbol}: Actual vs Predicted Excess Returns')
ax.set_ylabel('Excess Return')
ax.set_xlabel('Date')
ax.legend()
ax.grid(True, linestyle=':', linewidth=0.5)
plt.tight_layout()